# Tema de programare: Rezolvarea inconsistentei


Bun venit la tema despre **Rezolvarea inconsistentei**!

Datele din lumea reala sunt dezordonate - aceeasi entitate poate fi stocata in zeci de moduri diferite. Ia in considerare aceste tipare comune:

| Tip de coloana | Exemple de inconsistente |
|-------------|------------------------|
| **Nume de tari** | `'USA'`, `'US'`, `'U.S.A'`, `'United States'`, `'united states of america'` |
| **Valori monetare** | `'$1,200.50'`, `'900.00 USD'`, `'1200'`, `'€800'` |
| **Formate de data** | `'2023-01-15'`, `'01/15/2023'`, `'January 15, 2023'`, `'15-Jan-23'` |
| **Greutate / unitati** | `'72.5kg'`, `'80 KG'`, `'65'`, `'90.1 Kg'` |
| **Text categorial** | `'Active'`, `'ACTIVE'`, `'active'`, `'aCtIvE'` |

Fara rezolvarea acestor inconsistente, analizele tale vor produce in tacere rezultate gresite - group-by-urile se vor fragmenta, merge-urile vor rata potriviri, iar modelele vor trata valorile echivalente ca si categorii distincte.

**Ce vei face in aceasta tema:**

* Vei implementa **standardizarea numelor de tari** pentru a unifica toate variantele USA
* Vei implementa **parsarea valorilor monetare** pentru a extrage valori numerice din siruri dezordonate
* Vei implementa **standardizarea formatului datei** pentru a produce uniform output `YYYY-MM-DD`
* Vei implementa **parsarea greutatii** pentru a elimina sufixele de unitate si a converti la float
* Vei implementa **standardizarea literei de caz** pentru a comprima etichetele categoriale cu litere mari/mici amestecate

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI TALE:</h4>

* Toate celulele sunt inghetate, cu exceptia celor in care trebuie sa trimiti solutiile tale sau atunci cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde trebuie sa scrii codul solutiei. **Nu adauga si nu modifica niciun cod care este in afara acestor comentarii**.

* Poti adauga celule noi pentru a experimenta, dar acestea vor fi omise de evaluator, asa ca nu te baza pe celulele nou create pentru a gazdui codul solutiei tale; foloseste locurile oferite pentru asta.

---


## Cuprins
- [Importuri](#imports)
- [1 - Intelegerea datelor](#1---understanding-the-data)
- [2 - Standardizarea numelor de tari](#2---country-name-standardisation)
    - **[Exercitiul 1 - `standardize_country`](#exercise-1---standardize_country)**
- [3 - Parsarea valorilor monetare](#3---currency-parsing)
    - **[Exercitiul 2 - `parse_currency`](#exercise-2---parse_currency)**
- [4 - Standardizarea formatului datei](#4---date-format-standardisation)
    - **[Exercitiul 3 - `standardize_date_format`](#exercise-3---standardize_date_format)**
- [5 - Parsarea greutatii](#5---weight-parsing)
    - **[Exercitiul 4 - `parse_weight`](#exercise-4---parse_weight)**
- [6 - Standardizarea literei de caz in text](#6---text-case-standardisation)
    - **[Exercitiul 5 - `standardize_text_case`](#exercise-5---standardize_text_case)**
- [7 - Vizualizarea rezultatelor](#7---visualizing-results)


<a name='imports'></a>
## Importuri


In [ ]:
import re
import numpy as np
import pandas as pd

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-the-data'></a>
## 1 - Intelegerea datelor

Generam un set de date sintetic cu **5 coloane**, fiecare prezentand un tip diferit de inconsistenta:

| Coloana | Tip de inconsistenta |
|--------|-------------------|
| `country` | Scrieri si abrevieri amestecate pentru numele tarilor |
| `price` | Siruri monetare cu simboluri, virgule si sufixe |
| `date` | Siruri de data in formate multiple |
| `weight_kg` | Valori numerice cu sufixe de unitate variate |
| `status` | Etichete categoriale cu litere mari/mici inconsistente |

Ruleaza celulele de mai jos pentru a explora datele brute inainte sa incepi curatarea lor.


In [ ]:
df = helper_utils.generate_messy_dataset(n_samples=200, random_state=42)
print(f"Shape: {df.shape}")
print(f"\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique — sample: {df[col].unique()[:5].tolist()}")

In [ ]:
helper_utils.plot_inconsistency_summary(df)

<a name='2---country-name-standardisation'></a>
## 2 - Standardizarea numelor de tari

<a name='exercise-1---standardize_country'></a>
### **Exercitiul 1 - `standardize_country`**

**Sarcina ta:**
Implementeaza o functie care mapeaza toate variantele recunoscute ale numelui USA la sirul canonic `'United States'`.

**Cerinte:**
- Accepta un `pd.Series` de siruri cu nume de tari.
- Returneaza un nou `pd.Series` in care fiecare varianta USA este inlocuita cu `'United States'`.
- Valorile care nu sunt USA trebuie returnate neschimbate.
- Potrivirea trebuie sa fie **case-insensitive** si sa elimine whitespace-ul de la inceput/sfarsit.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Defineste un set de variante USA lowercase: `{'usa', 'us', 'u.s.a', 'u.s.', 'united states', 'united states of america', ...}`.
2. Foloseste `series.apply(lambda x: ...)` pentru a procesa fiecare valoare.
3. In interiorul lambda, normalizeaza valoarea: `str(x).lower().strip()`.
4. Verifica apartenenta in setul tau de variante - daca exista potrivire, returneaza `'United States'`; altfel returneaza `x`.
5. In plus, normalizeaza prin eliminarea punctelor si spatiilor pentru a prinde variante precum `'U.S.A.'` → `'usa'`.

</details>


In [ ]:
# GRADED FUNCTION: standardize_country
def standardize_country(series):
    """
    Standardize country name variants to a canonical form.

    Args:
        series (pd.Series): Series with country name strings.

    Returns:
        pd.Series: Series with standardized country names.
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
country_clean = standardize_country(df['country'])
print(f"Before unique values: {df['country'].unique().tolist()}")
print(f"After unique values:  {country_clean.unique().tolist()}")

In [ ]:
unittests.exercise_1(standardize_country)

<a name='3---currency-parsing'></a>
## 3 - Parsarea valorilor monetare

<a name='exercise-2---parse_currency'></a>
### **Exercitiul 2 - `parse_currency`**

**Sarcina ta:**
Implementeaza o functie care extrage valoarea numerica din siruri monetare dezordonate precum `'$1,200.50'`, `'900.00 USD'` sau `'€1500'`.

**Cerinte:**
- Accepta un `pd.Series` de siruri monetare.
- Returneaza un `pd.Series` de valori `float`.
- Valorile care nu pot fi parsate trebuie sa devina `NaN`.
- Gestioneaza elegant inputurile `NaN` (lasa-le sa treaca drept `NaN`).

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Defineste un helper intern `_parse(val)` care proceseaza un singur sir.
2. Returneaza `np.nan` imediat daca `pd.isna(val)`.
3. Foloseste `re.sub(r'[^\d.]', '', str(val))` pentru a elimina totul in afara cifrelor si punctelor.
4. Incearca `float(cleaned)` - daca `cleaned` este gol sau conversia esueaza, returneaza `np.nan`.
5. Aplica helper-ul cu `series.apply(_parse)`.

</details>


In [ ]:
# GRADED FUNCTION: parse_currency
def parse_currency(series):
    """
    Extract numeric values from currency strings.

    Args:
        series (pd.Series): Series with currency strings (e.g. '$1,200.50', '900 USD').

    Returns:
        pd.Series: Float Series with extracted values; unparseable values become NaN.
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
price_clean = parse_currency(df['price'])
print(f"Before sample: {df['price'].head(3).tolist()}")
print(f"After sample:  {price_clean.head(3).tolist()}")
print(f"dtype: {price_clean.dtype}")

In [ ]:
unittests.exercise_2(parse_currency)

<a name='4---date-format-standardisation'></a>
## 4 - Standardizarea formatului datei

<a name='exercise-3---standardize_date_format'></a>
### **Exercitiul 3 - `standardize_date_format`**

**Sarcina ta:**
Parseaza sirurile de data care apar in mai multe formate si afiseaza-le uniform ca siruri `'YYYY-MM-DD'`.

**Cerinte:**
- Accepta un `pd.Series` de siruri de data in formate mixte.
- Accepta un parametru `output_format` (implicit `'%Y-%m-%d'`).
- Returneaza un `pd.Series` de siruri formatate conform `output_format`.
- Datele care nu pot fi parsate trebuie sa devina `NaN` (nu sa arunce exceptie).

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Foloseste `pd.to_datetime(series, format='mixed', errors='coerce')` - `format='mixed'` (pandas ≥ 2.0) deduce formatul independent pentru fiecare valoare, gestionand un Series care amesteca de exemplu `'2023-01-15'` si `'01/15/2023'`.
2. Rezultatul este un Series datetime; intrarile invalide devin `NaT`.
3. Apeleaza `.dt.strftime(output_format)` pentru a converti datele valide in siruri.
4. Foloseste `.where(parsed.notna(), other=np.nan)` pentru a inlocui pozitiile derivate din `NaT` cu `NaN` real.

</details>


In [ ]:
# GRADED FUNCTION: standardize_date_format
def standardize_date_format(series, output_format='%Y-%m-%d'):
    """
    Parse date strings in various formats and output them uniformly.

    Args:
        series (pd.Series): Series with date strings in mixed formats.
        output_format (str): Target strftime format (default '%Y-%m-%d').

    Returns:
        pd.Series: String Series in output_format; invalid dates become NaN.
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
date_clean = standardize_date_format(df['date'])
print(f"Before sample: {df['date'].head(3).tolist()}")
print(f"After sample:  {date_clean.head(3).tolist()}")

In [ ]:
unittests.exercise_3(standardize_date_format)

<a name='5---weight-parsing'></a>
## 5 - Parsarea greutatii

<a name='exercise-4---parse_weight'></a>
### **Exercitiul 4 - `parse_weight`**

**Sarcina ta:**
Parseaza sirurile de greutate eliminand sufixele unitatilor (de ex. `'kg'`, `'KG'`, `'Kg'`) si convertind partea numerica ramasa la `float`.

**Cerinte:**
- Accepta un `pd.Series` de siruri de greutate precum `'72.5kg'`, `'80 KG'`, `'65'`.
- Returneaza un `pd.Series` de valori `float`.
- Sirurile goale si valorile care nu pot fi parsate trebuie sa devina `NaN`.
- Gestioneaza elegant inputurile `NaN`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Defineste un helper intern `_parse(val)`.
2. Returneaza `np.nan` imediat daca `pd.isna(val)`.
3. Foloseste `re.sub(r'[a-zA-Z\s]', '', str(val)).strip()` pentru a elimina toate literele si whitespace-ul - astfel ramane doar partea numerica.
4. Incearca `float(cleaned)` - daca `cleaned` este gol sau conversia esueaza, returneaza `np.nan`.
5. Aplica cu `series.apply(_parse)`.

</details>


In [ ]:
# GRADED FUNCTION: parse_weight
def parse_weight(series):
    """
    Parse weight values by stripping unit suffixes and converting to float.

    Args:
        series (pd.Series): Series with weight strings e.g. '72.5kg', '80 KG', '65'.

    Returns:
        pd.Series: Float Series; empty strings or unparseable values become NaN.
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
weight_clean = parse_weight(df['weight_kg'])
print(f"Before sample: {df['weight_kg'].head(3).tolist()}")
print(f"After sample:  {weight_clean.head(3).tolist()}")

In [ ]:
unittests.exercise_4(parse_weight)

<a name='6---text-case-standardisation'></a>
## 6 - Standardizarea literei de caz in text

<a name='exercise-5---standardize_text_case'></a>
### **Exercitiul 5 - `standardize_text_case`**

**Sarcina ta:**
Standardizeaza literele mari/mici pentru toate valorile de tip sir dintr-un Series la un format uniform - util pentru a comprima `'Active'`, `'ACTIVE'` si `'active'` intr-o singura eticheta canonica.

**Cerinte:**
- Accepta un `pd.Series` de valori sir si un parametru `case` (`'lower'`, `'upper'` sau `'title'`).
- Returneaza un nou `pd.Series` cu toate valorile convertite la litera de caz specificata.
- Daca este transmis un `case` nerecunoscut, returneaza o copie neschimbata a Series-ului original.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Foloseste `series.str.lower()` pentru lowercase, `series.str.upper()` pentru uppercase si `series.str.title()` pentru title case.
2. Un lant simplu `if/elif/elif/else` bazat pe parametrul `case` este cea mai curata abordare.
3. Pentru fallback, returneaza `series.copy()` pentru a evita modificarea originalului.

</details>


In [ ]:
# GRADED FUNCTION: standardize_text_case
def standardize_text_case(series, case='lower'):
    """
    Standardize string casing across a Series.

    Args:
        series (pd.Series): Series with string values.
        case (str): Target case: 'lower', 'upper', or 'title'.

    Returns:
        pd.Series: Series with uniform casing.
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
status_clean = standardize_text_case(df['status'], case='lower')
print(f"Before unique: {sorted(df['status'].unique())}")
print(f"After unique:  {sorted(status_clean.unique())}")

In [ ]:
unittests.exercise_5(standardize_text_case)

<a name='7---visualizing-results'></a>
## 7 - Vizualizarea rezultatelor


In [ ]:
helper_utils.visualize_before_after(df['country'], standardize_country(df['country']), 'country')
helper_utils.visualize_before_after(df['status'], standardize_text_case(df['status']), 'status')